# 06 - Feature Engineering

## Customer Intelligence Platform

---
This notebook creates engineered features that separate this project from
average analyses. Each feature group has a clear business rationale.

### Feature Groups
1. Tenure Features
2. Service Count
3. Revenue Features
4. Customer Profile Features
5. Risk Indicators
6. Interaction Features

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.load_data import load_clean
# from src.feature_engineering import (
#     create_tenure_features,
#     create_service_count,
#     create_revenue_features,
#     create_customer_profile_features,
#     create_risk_indicators,
#     create_interaction_features,
#     engineer_features,
# )
from src.feature_engineering import *
from src.config import TARGET, FEATURES_FILE
from src.utils import df_summary, save_dataframe

%matplotlib inline

In [2]:
df = load_clean()
df_summary(df, "Clean Data (Input)")


✅ Loaded clean data: 7,043 rows × 20 cols

[INFO] Clean Data (Input)
   Shape: 7,043 rows x 20 columns

Column Types:
 str        16
int64       2
float64     2
Name: count, dtype: int64

   Memory: 1.70 MB
   Duplicate rows: 22


##  Tenure Features

---
**Business Rationale**: 
- Raw tenure in months hides important behavioral patterns.
- Bucketing reveals distinct customer lifecycle stages.


In [3]:
df = create_tenure_features(df)
print("Tenure Group Distribution:")
print(df["tenure_group"].value_counts().sort_index())
print(f"\nNew Customers (<12 months): {df['is_new_customer'].sum():,} ({df['is_new_customer'].mean():.1%})")
print(f"\nChurn by Tenure Group:")
print(df.groupby("tenure_group", observed=True)[TARGET].mean().to_string())

Tenure Group Distribution:
tenure_group
0-12     2186
12-24    1024
24-48    1594
48+      2239
Name: count, dtype: int64

New Customers (<12 months): 2,069 (29.4%)

Churn by Tenure Group:
tenure_group
0-12     0.474382
12-24    0.287109
24-48    0.203890
48+      0.095132


## Service Count

---
**Business Rationale**: 
- Customers with more services are stickier.
- The total count captures engagement depth.


In [4]:
df = create_service_count(df)
print("Service Count Distribution:")
print(df["service_count"].value_counts().sort_index())
print(f"\nChurn by Service Count:")
print(df.groupby("service_count")[TARGET].mean().to_string())

Service Count Distribution:
service_count
0      80
1    1701
2    1188
3     965
4     922
5     908
6     676
7     395
8     208
Name: count, dtype: int64

Churn by Service Count:
service_count
0    0.437500
1    0.211052
2    0.328283
3    0.364767
4    0.313449
5    0.255507
6    0.224852
7    0.124051
8    0.052885


## Revenue Features

---
**Business Rationale**: 
- Revenue per month and revenue intensity capture spending patterns beyond raw charge amounts.


In [5]:
df = create_revenue_features(df)
print(f"Revenue per Month: mean={df['revenue_per_month'].mean():.2f}, std={df['revenue_per_month'].std():.2f}")
print(f"Revenue Intensity: mean={df['revenue_intensity'].mean():.2f}, std={df['revenue_intensity'].std():.2f}")

Revenue per Month: mean=64.76, std=30.19
Revenue Intensity: mean=1.00, std=0.05


## Customer Profile Features

---
**Business Rationale**: 
- Family status, tech adoption, and service bundle provide higher-level customer characterization.


In [6]:
df = create_customer_profile_features(df)
print(f"Has Family: {df['has_family'].sum():,} ({df['has_family'].mean():.1%})")
print(f"\nTech Adoption Score Distribution:")
print(df["tech_adoption_score"].value_counts().sort_index())
print(f"\nService Bundle Distribution:")
print(df["service_bundle"].value_counts())


Has Family: 3,704 (52.6%)

Tech Adoption Score Distribution:
tech_adoption_score
0    2793
1    1467
2    1372
3     941
4     470
Name: count, dtype: int64

Service Bundle Distribution:
service_bundle
Standard    2153
Premium     1830
Basic       1781
All-In      1279
Name: count, dtype: int64


## Risk Indicators

---
**Business Rationale**: 
- Combining known risk factors into a composite score creates a simple risk stratification.


In [7]:
df = create_risk_indicators(df)
print("Risk Score Distribution:")
print(df["risk_score"].value_counts().sort_index())
print(f"\nChurn by Risk Score:")
print(df.groupby("risk_score")[TARGET].mean().to_string())

Risk Score Distribution:
risk_score
0    1830
1    1518
2    1804
3    1285
4     606
Name: count, dtype: int64

Churn by Risk Score:
risk_score
0    0.031148
1    0.106061
2    0.292129
3    0.536187
4    0.717822


The risk score shows a clear monotonic relationship with churn:   
**higher risk scores mean higher churn rates**. This validates our feature design.

## Interaction Features

---
**Business Rationale**: 
- Capture non-linear relationships between features.

In [8]:
df = create_interaction_features(df)
print(f"Security Gap (Fiber without security): {df['security_gap'].sum():,} customers")
print(f"  Churn rate: {df[df['security_gap']==1][TARGET].mean():.1%}")
print(f"\nHigh Charge New Customer: {df['high_charge_new_customer'].sum():,} customers")
print(f"  Churn rate: {df[df['high_charge_new_customer']==1][TARGET].mean():.1%}")

Security Gap (Fiber without security): 2,257 customers
  Churn rate: 49.4%

High Charge New Customer: 781 customers
  Churn rate: 69.7%


## Feature Summary

---

In [9]:
df_summary(df, "Feature-Engineered Data")
print(f"\nNew columns added: {len(df.columns) - 20}")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")


[INFO] Feature-Engineered Data
   Shape: 7,043 rows x 35 columns

Column Types:
 str         16
int64       13
float64      4
category     1
category     1
Name: count, dtype: int64

   Memory: 2.41 MB
   Duplicate rows: 22

New columns added: 15

All columns (35):
  1. Gender
  2. Senior Citizen
  3. Partner
  4. Dependents
  5. Tenure Months
  6. Phone Service
  7. Multiple Lines
  8. Internet Service
  9. Online Security
  10. Online Backup
  11. Device Protection
  12. Tech Support
  13. Streaming TV
  14. Streaming Movies
  15. Contract
  16. Paperless Billing
  17. Payment Method
  18. Monthly Charges
  19. Total Charges
  20. Churn Value
  21. tenure_group
  22. is_new_customer
  23. service_count
  24. revenue_per_month
  25. revenue_intensity
  26. has_family
  27. tech_adoption_score
  28. service_bundle
  29. is_month_to_month
  30. is_fiber
  31. is_electronic_check
  32. risk_score
  33. contract_tenure
  34. security_gap
  35. high_charge_new_customer


## Validate No Leakage

---

In [10]:
# Check engineered features for target leakage
from src.config import LEAKAGE_COLUMNS
leaked = [c for c in LEAKAGE_COLUMNS if c in df.columns]
print(f"Leakage columns present: {leaked or 'None - All clear!'}")
print(f"\nTarget column present: {TARGET in df.columns}")

Leakage columns present: None - All clear!

Target column present: True


## Save Engineered Features

---

In [11]:
save_dataframe(df, FEATURES_FILE)
print(f"\nSaved to: {FEATURES_FILE}")

[OK] Saved 7,043 rows x 35 cols -> telco_features.csv

Saved to: S:\customer-intelligence-platform\data\processed\telco_features.csv


## Summary

| Feature Group | Features Created | Business Value |
|---------------|-----------------|----------------|
| Tenure | tenure_group, is_new_customer | Customer lifecycle stage |
| Service Count | service_count | Engagement depth |
| Revenue | revenue_per_month, revenue_intensity | Spending patterns |
| Customer Profile | has_family, tech_adoption_score, service_bundle | Customer characterization |
| Risk Indicators | is_m2m, is_fiber, is_e_check, risk_score | Risk stratification |
| Interactions | contract_tenure, security_gap, high_charge_new | Non-linear patterns |

**Total: 15 new features** added to the 20 clean features.
